<a href="https://colab.research.google.com/github/John-588-git/Dashboard/blob/main/LANGUAGE%20MODELLING%20AND%20SEQUENCE%20TAGGING.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
#Task 1: Language Modeling – Next Word Prediction
#1. Install the required libraries
!pip install nltk spacy scikit-learn pandas matplotlib
!python -m spacy download en_core_web_sm
#2. Import libraries and prepare the text
import re
import math
import pandas as pd
from collections import Counter, defaultdict

import nltk
from nltk.tokenize import word_tokenize

nltk.download('punkt')
nltk.download('punkt_tab')
#sample preprocessed text to use.
text = """
The cat sat on the mat.
The cat ate the food.
The cat slept on the mat.
The dog sat on the mat.
The dog ate the food.
The dog slept on the floor.
Machine learning models learn from data.
Machine learning models can make predictions.
Machine learning algorithms learn patterns from data.
Machine learning models are useful for prediction.
"""

# Convert to lowercase and tokenize
text = text.lower()

sentences = [s.strip() for s in text.split(".") if s.strip()]

tokenized_sentences = []

for sentence in sentences:
    tokens = word_tokenize(sentence)
    tokens = ["<START>"] + tokens + ["<END>"]
    tokenized_sentences.append(tokens)

tokenized_sentences
"""3. Building a trigram language model

A trigram model predicts the next word using the previous two words.

For example:

"The cat" → "sat"

The probability is:  P(w3|w1,w2)=Count(w1,w2,w3)/Count(w1,w2)"""
# Count bigrams and trigrams
bigram_counts = Counter()
trigram_counts = Counter()

for tokens in tokenized_sentences:

    for i in range(len(tokens) - 1):
        bigram = (tokens[i], tokens[i + 1])
        bigram_counts[bigram] += 1

    for i in range(len(tokens) - 2):
        trigram = (tokens[i], tokens[i + 1], tokens[i + 2])
        trigram_counts[trigram] += 1

print("Number of bigrams:", len(bigram_counts))
print("Number of trigrams:", len(trigram_counts))

#4. Calculating next-word probabilities.
def next_word_probabilities(word1, word2):
    """
    Calculate probabilities of possible next words
    given the previous two words.
    """

    word1 = word1.lower()
    word2 = word2.lower()

    denominator = bigram_counts[(word1, word2)]

    if denominator == 0:
        return {}

    probabilities = {}

    for (w1, w2, w3), count in trigram_counts.items():

        if w1 == word1 and w2 == word2:
            probabilities[w3] = count / denominator

    return probabilities

#Testing the next-word probabilities
probabilities = next_word_probabilities("the", "cat")

print("Next-word probabilities:")

for word, probability in sorted(
    probabilities.items(),
    key=lambda x: x[1],
    reverse=True
):
    print(f"{word}: {probability:.3f}")

#5. Predicting the next word
def predict_next_word(sequence):
    """
    Predict the most probable next word
    using the last two words in the sequence.
    """

    words = word_tokenize(sequence.lower())

    if len(words) < 2:
        return "Please provide at least two words."

    word1 = words[-2]
    word2 = words[-1]

    probabilities = next_word_probabilities(word1, word2)

    if not probabilities:
        return "No prediction available."

    predicted_word = max(
        probabilities,
        key=probabilities.get
    )

    return predicted_word

#Testing the model.
test_sentences = [
    "the cat",
    "the dog",
    "machine learning",
    "learning models"
]

for sentence in test_sentences:

    prediction = predict_next_word(sentence)

    print(
        f"Input: {sentence} -> "
        f"Predicted next word: {prediction}"
    )


#Task 2: Sequence Tagging
"""A. POS Tagging

POS means Part-of-Speech tagging. It assigns grammatical categories such as:

DET — determiner
NOUN — noun
VERB — verb
ADJ — adjective
ADV — adverb
PROPN — proper noun
PRON — pronoun"""

import spacy

nlp = spacy.load("en_core_web_sm")
#Testing POS tagging:
sentence = "The cat sleeps on the mat."

doc = nlp(sentence)

print("POS TAGGING RESULTS\n")

for token in doc:
    print(
        f"{token.text:12} "
        f"{token.pos_:8} "
        f"{token.tag_}"
    )
    # ALIGHNING THE FORMAT
    print("\nFormatted POS tagging:")

for token in doc:
    print(f"{token.text}/{token.pos_}", end=" ")


"""B. Named Entity Recognition (NER)

NER identifies named entities such as:

PERSON
GPE — geographical/political entity
ORG — organization
DATE
MONEY
LOC"""

#sentence =
"""
Barack Obama was the 44th president of the United States.
He was born in Hawaii in 1961.
"""

doc = nlp(sentence)

print("NAMED ENTITY RECOGNITION\n")

for entity in doc.ents:
    print(
        f"Entity: {entity.text:20} "
        f"Type: {entity.label_:10} "
        f"Description: {spacy.explain(entity.label_)}"
    )

#ENTITIES IN READABLE FORM
    print("\nFormatted NER results:")

for entity in doc.ents:
    print(f"{entity.text}/{entity.label_}")


#Task 3: EVALUATION

"""A. Language Model Evaluation

One useful evaluation measure for language models is perplexity.

Perplexity measures how well a language model predicts a sequence. Lower perplexity generally indicates better predictive performance.

The formula is:
PP(W)=exp(-(1/N)*Σlog(P(w_i|w_(i-2),w_(i-1))))"""

#CalculatING trigram perplexity
def calculate_perplexity(sentence):

    tokens = word_tokenize(sentence.lower())

    if len(tokens) < 3:
        return None

    log_probability = 0
    count = 0

    for i in range(2, len(tokens)):

        w1 = tokens[i - 2]
        w2 = tokens[i - 1]
        w3 = tokens[i]

        trigram = (w1, w2, w3)
        bigram = (w1, w2)

        trigram_count = trigram_counts[trigram]
        bigram_count = bigram_counts[bigram]

        if trigram_count == 0 or bigram_count == 0:
            return float("inf")

        probability = trigram_count / bigram_count

        log_probability += math.log(probability)
        count += 1

    perplexity = math.exp(-log_probability / count)

    return perplexity
#TESTING TRIAGRAM PERPLEXITY
test_sentence = "the cat sat on the mat"

pp = calculate_perplexity(test_sentence)

print("Sentence:", test_sentence)
print("Perplexity:", pp)

#PREDICTION ACCURACY
def prediction_accuracy(sentences):

    correct = 0
    total = 0

    for sentence in sentences:

        words = word_tokenize(sentence.lower())

        for i in range(2, len(words)):

            context = f"{words[i-2]} {words[i-1]}"

            prediction = predict_next_word(context)

            actual = words[i]

            if prediction == actual:
                correct += 1

            total += 1

    if total == 0:
        return 0

    return correct / total

#THE ACCURACY
accuracy = prediction_accuracy(sentences)

print(f"Language Model Accuracy: {accuracy:.2%}")

#B. POS Tagging Evaluation-(Demontration)

"""If you have gold-standard POS tags, accuracy can be calculated as:

Accuracy = Total Tags/Correct Tags"""
from sklearn.metrics import accuracy_score

# Example predicted and actual POS tags
actual_tags = [
    "DET", "NOUN", "VERB",
    "ADP", "DET", "NOUN"
]

predicted_tags = [
    "DET", "NOUN", "VERB",
    "ADP", "DET", "NOUN"
]

pos_accuracy = accuracy_score(
    actual_tags,
    predicted_tags
)

print(f"POS Tagging Accuracy: {pos_accuracy:.2%}")


#C. NER Evaluation

"""For NER, use:

Precision
Recall
F1-score
The formulas are:
Precision=TP/(TP+FP)
	​Recall=TP/(FN+TP)
	F1=2×Precision*Recall/Precision+Recall"""
from sklearn.metrics import classification_report

actual_ner = [
    "PERSON",
    "O",
    "O",
    "GPE"
]

predicted_ner = [
    "PERSON",
    "O",
    "O",
    "GPE"
]

print(
    classification_report(
        actual_ner,
        predicted_ner,
        zero_division=0
    )
)

"""#The complete workflow is:
                 PREPROCESSED TEXT
                         |
             +-----------+-----------+
             |                       |
             v                       v
       LANGUAGE MODEL          SEQUENCE TAGGING
             |                       |
       Trigram Model            +----+----+
             |                  |         |
             v                  v         v
      Next-word prediction     POS       NER
             |                  |         |
             v                  v         v
        Probability          Accuracy  P/R/F1
             |
             v
        Perplexity"""

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 48.1 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


Number of bigrams: 39
Number of trigrams: 42
Next-word probabilities:
sat: 0.333
ate: 0.333
slept: 0.333
Input: the cat -> Predicted next word: sat
Input: the dog -> Predicted next word: sat
Input: machine learning -> Predicted next word: models
Input: learning models -> Predicted next word: learn
POS TAGGING RESULTS

The          DET      DT

Formatted POS tagging:
cat          NOUN     NN

Formatted POS tagging:
sleeps       VERB     VBZ

Formatted POS tagging:
on           ADP      IN

Formatted POS tagging:
the          DET      DT

Formatted POS tagging:
mat          NOUN     NN

Formatted POS tagging:
.            PUNCT    .

Formatted POS tagging:
The/DET cat/NOUN sleeps/VERB on/ADP the/DET mat/NOUN ./PUNCT NAMED ENTITY RECOGNITION

Sentence: the cat sat on the mat
Perplexity: 1.4142135623730951
Language Model Accuracy: 80.00%
POS Tagging Accuracy: 100.00%
              precision    recall  f1-score   support

         GPE       1.00      1.00      1.00         1
           O   

'#The complete workflow is:\n                 PREPROCESSED TEXT\n                         |\n             +-----------+-----------+\n             |                       |\n             v                       v\n       LANGUAGE MODEL          SEQUENCE TAGGING\n             |                       |\n       Trigram Model            +----+----+\n             |                  |         |\n             v                  v         v\n      Next-word prediction     POS       NER\n             |                  |         |\n             v                  v         v\n        Probability          Accuracy  P/R/F1\n             |\n             v\n        Perplexity'